# 27. Colab GPU Smoke Test

Notebook de verificación rápida del pipeline `qml4var` sobre la GPU de Google Colab,
antes de lanzar experimentos al servidor de GPU dedicado.

**Qué comprueba:**
- CUDA disponible y visible por PyTorch.
- Instalación correcta de PennyLane.
- Construcción del ansatz `hardware_efficient_ansatz` y creación del `workflow_cfg`.
- Entrenamiento **supervisado** y **no supervisado** con 3 strikes × 3 repeticiones.
- Cálculo de precio por Fourier y visualización de resultados.

**Arquitectura de aceleración:**
- Circuitos: `default.qubit` + `backprop` (necesario para derivadas de segundo orden del término PDF).
- Batching: `qml.vmap` evalúa todos los puntos de entrenamiento en una sola llamada al circuito (elimina el bucle Python por muestra).
- GPU: `torch_device="cuda"` — todas las operaciones de tensores PyTorch (Adam, loss, gradientes) corren en CUDA.

---

**Antes de ejecutar:** asegúrate de que el entorno de ejecución de Colab
tiene acelerador GPU activo (`Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`).

## 0. Verificar GPU del entorno Colab

In [3]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("[WARN] nvidia-smi no disponible. Verifica que el entorno tiene GPU activa.")
    print(result.stderr)

Sun Mar 22 16:15:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Instalar dependencias

El pipeline usa `default.qubit` + `backprop` + `qml.vmap` para batching.
La GPU acelera las operaciones de tensores PyTorch vía `torch_device="cuda"`.
`pennylane-lightning[gpu]` **no es necesario**.

In [4]:
import sys

!{sys.executable} -m pip install --quiet \
    "pennylane>=0.38" \
    "pennylane-lightning>=0.38" \
    numpy

print("Instalación completada.")

Instalación completada.


## 2. Montar el repositorio

Dos opciones; ejecuta **sólo una** de las dos celdas siguientes.

### Opción A — Google Drive (recomendada si el repo está en Drive)

In [5]:
import os

from google.colab import drive

drive.mount("/content/drive")

# Ajusta la ruta si el repo está en otro lugar de tu Drive
REPO_ROOT = "/content/drive/MyDrive/CodeTFM"

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
print("REPO_ROOT:", REPO_ROOT)


Mounted at /content/drive
REPO_ROOT: /content/drive/MyDrive/CodeTFM


### Opción B — Clonar desde GitHub

In [6]:
# Sustituye la URL por la de tu repositorio
# !git clone https://github.com/TU_USUARIO/CodeTFM.git /content/CodeTFM

# REPO_ROOT = "/content/CodeTFM"
# import sys, os
# sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
# print("REPO_ROOT:", REPO_ROOT)


## 3. Imports y verificación de GPU en PyTorch

In [7]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import scipy.stats as stats
import torch

from finance import (
    bs_put_price,
    estimate_price_from_trained_pqc,
)
from qml4var.adam import adam_optimizer_loop
from qml4var.architectures import (
    hardware_efficient_ansatz,
    init_weights,
    normalize_data,
)
from qml4var.data_utils import (
    empirical_cdf,
    simulate_black_scholes_data_rescaled,
)
from qml4var.losses import torch_gradient
from qml4var.workflows import (
    qdml_loss_workflow,
    unsupervised_qdml_loss_workflow,
    workflow_for_cdf,
    workflow_for_pdf,
)

# --- Verificación CUDA ---
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch version    : {torch.__version__}")
print(f"pennylane version: {qml.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version     : {torch.version.cuda}")
print(f"TORCH_DEVICE     : {TORCH_DEVICE}")

if TORCH_DEVICE == "cpu":
    print()
    print("[WARN] No se detectó GPU. El test continúa en CPU, pero verifica")
    print("       que el entorno de Colab tiene acelerador GPU activado.")


torch version    : 2.10.0+cu128
pennylane version: 0.44.1
CUDA available   : True
GPU name         : Tesla T4
CUDA version     : 12.8
TORCH_DEVICE     : cuda


## 4. Configuración del experimento

In [8]:
# Black-Scholes
S0, r, sigma, T = 100.0, 0.10, 0.25, 1.0

STRIKES = [90, 100, 110]   # ITM, ATM, OTM
REPETITIONS = 3                # repeticiones por (modo, strike)
BASE_SEED = 2026

TRAIN_INTERVAL = (-2.0 * np.pi, 2.0 * np.pi)
ENCODING_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)
FEATURES_NUMBER = 1

SMOKE_CFG = {
    "n_qubits_by_feature": 2,
    "n_layers": 2,
    "n_train": 500,
    "n_test": 100,
    "epochs": 10,
    "learning_rate": 0.05,
    "beta1": 0.9,
    "beta2": 0.999,
    "tolerance": 1.0e-8,
    "n_counts_tolerance": 999,
    "print_step": 5,
    "empirical_shift": -0.5,
    "loss_weights_sup": [0.9, 0.1],
    "loss_weights_unsup": [0.2, 0.8],
}

K_TERMS = 8
GRID_POINTS_F = 512

total_runs = 2 * len(STRIKES) * REPETITIONS
print(f"Strikes      : {STRIKES}")
print(f"Repeticiones : {REPETITIONS}")
print(f"Runs totales : {total_runs}  (2 modos × {len(STRIKES)} strikes × {REPETITIONS} reps)")
print(f"Config       : {SMOKE_CFG}")

Strikes      : [90, 100, 110]
Repeticiones : 3
Runs totales : 18  (2 modos × 3 strikes × 3 reps)
Config       : {'n_qubits_by_feature': 2, 'n_layers': 2, 'n_train': 500, 'n_test': 100, 'epochs': 10, 'learning_rate': 0.05, 'beta1': 0.9, 'beta2': 0.999, 'tolerance': 1e-08, 'n_counts_tolerance': 999, 'print_step': 5, 'empirical_shift': -0.5, 'loss_weights_sup': [0.9, 0.1], 'loss_weights_unsup': [0.2, 0.8]}


## 5. Construir circuito y `workflow_cfg`

In [9]:
base_frecuency, shift_feature = normalize_data(
    [TRAIN_INTERVAL[0]] * FEATURES_NUMBER,
    [TRAIN_INTERVAL[1]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[0]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[1]] * FEATURES_NUMBER,
)

circuit_fn, weights_names, features_names = hardware_efficient_ansatz(
    features_number=FEATURES_NUMBER,
    n_qubits_by_feature=SMOKE_CFG["n_qubits_by_feature"],
    n_layers=SMOKE_CFG["n_layers"],
    base_frecuency=list(base_frecuency),
    shift_feature=list(shift_feature),
)

workflow_cfg = {
    "circuit_fn": circuit_fn,
    "weights_names": weights_names,
    "features_names": features_names,
    "torch_device": TORCH_DEVICE,
    "minval": [TRAIN_INTERVAL[0]],
    "maxval": [TRAIN_INTERVAL[1]],
    "points": 60,
}

print(f"Qubits totales : {SMOKE_CFG['n_qubits_by_feature'] * FEATURES_NUMBER}")
print(f"Parámetros     : {len(weights_names)}")
print(f"torch_device   : {TORCH_DEVICE}")

[hardware_efficient_ansatz] device=default.qubit  diff_method=backprop  wires=2
Qubits totales : 2
Parámetros     : 4
torch_device   : cuda


## 6. Funciones auxiliares

In [10]:
def _optimizer_kwargs():
    return dict(
        epochs=SMOKE_CFG["epochs"],
        learning_rate=SMOKE_CFG["learning_rate"],
        beta1=SMOKE_CFG["beta1"],
        beta2=SMOKE_CFG["beta2"],
        tolerance=SMOKE_CFG["tolerance"],
        n_counts_tolerance=SMOKE_CFG["n_counts_tolerance"],
        print_step=SMOKE_CFG["print_step"],
    )


def train_supervised(u_train, wf_cfg, weights_names):
    y_train = empirical_cdf(u_train).reshape((-1, 1)) - 0.5

    def loss_fn(w):
        return qdml_loss_workflow(
            w, u_train, y_train, dask_client=None,
            loss_weights=SMOKE_CFG["loss_weights_sup"], **wf_cfg,
        )

    def _inner_loss(w_, x_, y_):
        return qdml_loss_workflow(
            w_, x_, y_, dask_client=None,
            loss_weights=SMOKE_CFG["loss_weights_sup"], **wf_cfg,
        )

    def grad_fn(w, x, y):
        return torch_gradient(w, x, y, _inner_loss)

    return adam_optimizer_loop(
        weights_dict=init_weights(weights_names),
        loss_function=loss_fn,
        metric_function=None,
        gradient_function=grad_fn,
        batch_generator=[(u_train, y_train)],
        initial_time=0,
        **_optimizer_kwargs(),
    )


def train_unsupervised(u_train, wf_cfg, weights_names):
    dummy_y = np.zeros((u_train.shape[0], 1))

    def loss_fn(w):
        return unsupervised_qdml_loss_workflow(
            w, u_train, dask_client=None,
            empirical_shift=SMOKE_CFG["empirical_shift"],
            loss_weights=SMOKE_CFG["loss_weights_unsup"], **wf_cfg,
        )

    def _inner_loss(w_, x_, y_):
        return unsupervised_qdml_loss_workflow(
            w_, x_, dask_client=None,
            empirical_shift=SMOKE_CFG["empirical_shift"],
            loss_weights=SMOKE_CFG["loss_weights_unsup"], **wf_cfg,
        )

    def grad_fn(w, x, y):
        return torch_gradient(w, x, y, _inner_loss)

    return adam_optimizer_loop(
        weights_dict=init_weights(weights_names),
        loss_function=loss_fn,
        metric_function=None,
        gradient_function=grad_fn,
        batch_generator=[(u_train, dummy_y)],
        initial_time=0,
        **_optimizer_kwargs(),
    )


print("Funciones de entrenamiento definidas.")


Funciones de entrenamiento definidas.


## 7. Experimento principal — loop sobre modos, strikes y repeticiones

In [ ]:
results = []
global_t0 = perf_counter()
run_idx = 0
total_runs = 2 * len(STRIKES) * REPETITIONS
n_total = SMOKE_CFG["n_train"] + SMOKE_CFG["n_test"]

for mode_name in ["supervised", "unsupervised"]:
    for K_ in STRIKES:
        theor_price = bs_put_price(S0, K_, r, sigma, T)
        for rep in range(1, REPETITIONS + 1):
            run_idx += 1
            seed = BASE_SEED + 100_000 * rep + 1_000 * int(K_)

            _, u_all, x_min, x_max = simulate_black_scholes_data_rescaled(
                S0_=S0, r_=r, T_=T, sigma_=sigma, K_=K_, n_points=n_total, seed=seed
            )
            u_train = u_all[:SMOKE_CFG["n_train"]]
            u_test  = u_all[SMOKE_CFG["n_train"]:]

            t0 = perf_counter()
            if mode_name == "supervised":
                weights = train_supervised(u_train, workflow_cfg, weights_names)
            else:
                weights = train_unsupervised(u_train, workflow_cfg, weights_names)
            t1 = perf_counter()
            run_seconds = t1 - t0

            # CDF MSE en test
            y_test = empirical_cdf(u_test).reshape(-1, 1) - 0.5
            y_pred = workflow_for_cdf(
                weights, u_test, dask_client=None, **workflow_cfg
            )["y_predict_cdf"].reshape(-1, 1)
            cdf_mse = float(np.mean((y_pred - y_test) ** 2))

            # Precio Fourier (Method I)
            est_price = estimate_price_from_trained_pqc(
                weights=weights,
                artifacts={"workflow_cfg": workflow_cfg},
                K_=K_,
                x_min_raw=x_min,
                x_max_raw=x_max,
                train_interval=TRAIN_INTERVAL,
                risk_free_rate=r,
                delta_t=T,
                k_terms=K_TERMS,
                grid_points=GRID_POINTS_F,
            )

            results.append({
                "mode": mode_name,
                "strike": K_,
                "rep": rep,
                "estimated_price": est_price,
                "theoretical_price": theor_price,
                "price_error": abs(est_price - theor_price) if np.isfinite(est_price) else np.nan,
                "cdf_mse": cdf_mse,
                "seconds": run_seconds,
            })

            elapsed = perf_counter() - global_t0
            avg = elapsed / run_idx
            eta = (total_runs - run_idx) * avg
            print(
                f"[{run_idx:>2}/{total_runs}] mode={mode_name:<12} K={K_:>3}  rep={rep}"
                f"  price={est_price:>7.4f} (theor={theor_price:.4f})"
                f"  cdf_mse={cdf_mse:.4f}  t={run_seconds:.1f}s  ETA={eta / 60:.1f}min"
            )

results_df = pd.DataFrame(results)
print(f"\nTotal tiempo: {(perf_counter() - global_t0) / 60:.1f} min")
results_df


	 MSE at t=0: None
	 Iteracion: 0. Loss: 0.38798759608921696
	 MSE at t=5: None
	 Iteracion: 5. Loss: 0.1392937065682745
Maximum number of iterations achieved.
[ 1/18] mode=supervised   K= 90  rep=1  price=25.2054 (theor=2.5988)  cdf_mse=0.0579  t=296.6s  ETA=85.9min
	 MSE at t=0: None
	 Iteracion: 0. Loss: 0.9546892298401597


## 8. Resumen estadístico por modo y strike

In [ ]:
summary = results_df.groupby(["mode", "strike"], as_index=False).agg(
    theor_price=("theoretical_price", "first"),
    mean_price=("estimated_price", "mean"),
    std_price=("estimated_price", "std"),
    mean_error=("price_error", "mean"),
    mean_cdf_mse=("cdf_mse", "mean"),
    mean_seconds=("seconds", "mean"),
)
summary["std_price"] = summary["std_price"].fillna(0.0)
summary["rel_error_%"] = (summary["mean_error"] / summary["theor_price"] * 100).round(2)
summary

## 9. Visualización de resultados

In [ ]:
COLORS = {"supervised": "#377eb8", "unsupervised": "#e41a1c"}
MODES = ["supervised", "unsupervised"]
x_pos = np.arange(len(STRIKES))
bar_w = 0.35

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Smoke test GPU — {SMOKE_CFG['n_qubits_by_feature']} qubits/feature · "
    f"{SMOKE_CFG['n_layers']} layers · {SMOKE_CFG['epochs']} épocas · "
    f"{REPETITIONS} repeticiones",
    fontsize=13, fontweight="bold"
)

# ── Plot 1: Precio estimado vs teórico ────────────────────────────────────────
ax = axes[0, 0]
for i, mode in enumerate(MODES):
    df_m = summary[summary["mode"] == mode]
    offset = (i - 0.5) * bar_w
    ax.bar(
        x_pos + offset, df_m["mean_price"], bar_w,
        yerr=df_m["std_price"], capsize=5,
        color=COLORS[mode], alpha=0.8, label=mode,
    )
# Líneas teóricas
for j, K_ in enumerate(STRIKES):
    p_t = bs_put_price(S0, K_, r, sigma, T)
    ax.hlines(p_t, j - bar_w, j + bar_w, colors="black", linewidths=1.5,
              linestyles="--", label="Black-Scholes" if j == 0 else None)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"K={k}" for k in STRIKES])
ax.set_title("Precio estimado (media ± std) vs teórico")
ax.set_ylabel("Precio put")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# ── Plot 2: Error absoluto medio ──────────────────────────────────────────────
ax = axes[0, 1]
for i, mode in enumerate(MODES):
    df_m = summary[summary["mode"] == mode]
    offset = (i - 0.5) * bar_w
    ax.bar(
        x_pos + offset, df_m["mean_error"], bar_w,
        color=COLORS[mode], alpha=0.8, label=mode,
    )
ax.set_xticks(x_pos)
ax.set_xticklabels([f"K={k}" for k in STRIKES])
ax.set_title("Error absoluto medio de precio")
ax.set_ylabel("|precio estimado − teórico|")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# ── Plot 3: CDF MSE en test ───────────────────────────────────────────────────
ax = axes[1, 0]
for i, mode in enumerate(MODES):
    df_m = summary[summary["mode"] == mode]
    offset = (i - 0.5) * bar_w
    ax.bar(
        x_pos + offset, df_m["mean_cdf_mse"], bar_w,
        color=COLORS[mode], alpha=0.8, label=mode,
    )
ax.set_xticks(x_pos)
ax.set_xticklabels([f"K={k}" for k in STRIKES])
ax.set_title("CDF MSE en test (media)")
ax.set_ylabel("MSE")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# ── Plot 4: Tiempo de entrenamiento por run ───────────────────────────────────
ax = axes[1, 1]
for i, mode in enumerate(MODES):
    df_m = results_df[results_df["mode"] == mode]
    for j, K_ in enumerate(STRIKES):
        df_k = df_m[df_m["strike"] == K_]["seconds"].values
        xs = np.full(len(df_k), j + (i - 0.5) * bar_w)
        ax.scatter(xs, df_k, color=COLORS[mode], alpha=0.8, s=60,
                   label=mode if j == 0 else None)
        ax.hlines(df_k.mean(), j + (i - 0.5) * bar_w - bar_w / 3,
                  j + (i - 0.5) * bar_w + bar_w / 3,
                  colors=COLORS[mode], linewidths=2)
ax.set_xticks(x_pos)
ax.set_xticklabels([f"K={k}" for k in STRIKES])
ax.set_title("Tiempo por run (s) — puntos individuales + media")
ax.set_ylabel("Segundos")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---

## 10. Checklist final

| Check | Esperado |
|-------|----------|
| `nvidia-smi` imprime GPU | Sin errores |
| `TORCH_DEVICE = cuda` | `cuda` (no `cpu`) |
| Loop completo sin excepciones | 18 runs (2 modos × 3 strikes × 3 reps) |
| Precios estimados | Valores finitos (no NaN) |
| Gráficas renderizadas | 4 subplots visibles |

Si todos los checks pasan, el repositorio está listo para ejecutarse en el servidor de GPU dedicado.